# 07 — Train Gaussian-center L1 + circle validation plots

**Active:** L1-only with `gaussian_center` spatial loss (σ=32) and Gaussian-circle viz.

Set **`COMPONENT = "u"`** or **`"v"`** to train one field at a time (after splitting data — see config cell).

- **Loss:** `l1` — Gaussian-weighted L1 only, no perceptual / adversarial / warmups
- **Spatial loss:** `gaussian_center` — center errors weighted more than outer rings
- **Error panels:** `|GT − pred|` L1 maps (inferno, vmin=0)
- **Viz every 5 epochs:** `viz/samples_gauss_circles/` + `circle_l1_overview.png`

**Separate u/v training:** split combined export → `iD=1`, one row in circle plots.

**Rerun:** same `RUN_TAG` overwrites prior results.

Outputs: `Models/adv_no/{RUN_NAME}/`

---

_Commented in config: adv-NO (VGG + adversarial) for later._

In [1]:
import importlib
import json
import sys
from dataclasses import asdict
from pathlib import Path

import src.train.adv_no_train as adv_no_train_module
importlib.reload(adv_no_train_module)
from src.train.adv_no_train import AdvNOTrainConfig, run_adv_no_training

In [2]:
DT = 96
UV_PREFIX = "data/sample_16_19_smallfixwin_uv_dt96"

# Train one component: "u" | "v" | "uv" (both together, iD=2)
COMPONENT = "u"

if COMPONENT == "uv":
    DATA_PREFIX = UV_PREFIX
    CHANNEL_NAMES = ("u", "v")
    iD = 2
    RUN_TAG = "gauss_l1_circles_uv"
else:
    DATA_PREFIX = f"data/sample_16_19_smallfixwin_{COMPONENT}_dt96"
    CHANNEL_NAMES = (COMPONENT,)
    iD = 1
    RUN_TAG = f"gauss_l1_circles_{COMPONENT}"

RUN_BASE = "adv_no_uv_16_19_dt96"
RUN_NAME = f"{RUN_BASE}_{RUN_TAG}"
RUN_DIR = Path("Models/adv_no") / RUN_NAME

LOSS_MODE = "l1"
LOSS_METRIC = "components"
LOSS_SPATIAL_MODE = "gaussian_center"
LOSS_GAUSSIAN_SIGMA = 32.0

# --- adv-NO (commented — restore later) ---
# LOSS_MODE = "adv_no"
# RUN_TAG = "gauss_vgg_warm21_adv41"

BETA = 0.0
LAMBDA_PERC = 0.0
PERCEPTUAL_BACKEND = "vgg"
LPIPS_NET = "alex"
PERCEPTUAL_WEIGHTS = (1.0, 1.0, 1.0, 1.0, 1.0)
PERCEPTUAL_WARMUP_EPOCHS = 0
WARMUP_EPOCHS = 0
D_LR_RATIO = 0.5
D_SKIP_THRESHOLD = 0.2

VIZ_GAUSSIAN_CIRCLES = True
VIZ_GAUSSIAN_CIRCLE_RADII = (16.0, 32.0, 64.0)
VIZ_GAUSSIAN_INNER_CIRCLE_RADII = (32.0, 16.0)
VIZ_GAUSSIAN_FIG_SCALE = 2.6
VIZ_RESIDUAL_METRIC = "l1"

USE_COSINE_LR = True
DETACHED = False
SHOW_INLINE_PLOTS = False
PREVIEW_LOSS_MASK = True
PROTECT_EXISTING_RUN = False
DEVICE = "cuda:7"

VIZ_EVERY = 5
VIZ_SAMPLES = 1
VIZ_SEED = 42

In [ ]:
# Run once: split combined u/v export → single-channel prefixes
from pathlib import Path
import subprocess

u_sims = Path("data/sample_16_19_smallfixwin_u_dt96_sims.npy")
if not u_sims.exists():
    subprocess.run(
        [
            "python", "scripts/split_uv_data_prefix.py",
            UV_PREFIX,
            "--u-prefix", "data/sample_16_19_smallfixwin_u_dt96",
            "--v-prefix", "data/sample_16_19_smallfixwin_v_dt96",
        ],
        check=True,
    )
else:
    print(f"Split data OK ({u_sims})")

In [3]:
train_config = AdvNOTrainConfig(
    data_prefix=DATA_PREFIX,
    run_name=RUN_NAME,
    dt_hours=DT,
    epochs=100,
    batch_size=4,
    lr=2e-4,
    d_lr=1e-4,
    d_lr_follow_g=True,
    d_lr_ratio=D_LR_RATIO,
    d_skip_threshold=D_SKIP_THRESHOLD,
    use_cosine_lr=USE_COSINE_LR,
    iD=iD,
    device=DEVICE,
    loss_mode=LOSS_MODE,
    loss_metric=LOSS_METRIC,
    loss_spatial_mode=LOSS_SPATIAL_MODE,
    loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
    protect_existing_run=PROTECT_EXISTING_RUN,
    preview_loss_mask=PREVIEW_LOSS_MASK,
    beta=BETA,
    lambda_perc=LAMBDA_PERC,
    perceptual_backend=PERCEPTUAL_BACKEND,
    lpips_net=LPIPS_NET,
    perceptual_weights=PERCEPTUAL_WEIGHTS,
    perceptual_warmup_epochs=PERCEPTUAL_WARMUP_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    checkpoint_subdir="Models/adv_no",
    isolate_runs=True,
    channel_names=CHANNEL_NAMES,
    viz_every=VIZ_EVERY,
    viz_samples=VIZ_SAMPLES,
    viz_seed=VIZ_SEED,
    viz_residual_metric=VIZ_RESIDUAL_METRIC,
    viz_gaussian_circles=VIZ_GAUSSIAN_CIRCLES,
    viz_gaussian_circle_radii=VIZ_GAUSSIAN_CIRCLE_RADII,
    viz_gaussian_inner_circle_radii=VIZ_GAUSSIAN_INNER_CIRCLE_RADII,
    viz_gaussian_fig_scale=VIZ_GAUSSIAN_FIG_SCALE,
    show_inline_plots=SHOW_INLINE_PLOTS,
)

RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / f"train_config_{RUN_NAME}.json"
config_path.write_text(json.dumps(asdict(train_config), indent=2, default=str))
print(f"Run directory: {RUN_DIR}")
print(f"Wrote config:  {config_path}")
print(f"  data: {DATA_PREFIX} | iD={iD} | loss_metric={LOSS_METRIC}")
if LOSS_MODE == "l1":
    print("  loss mode: L1-only (no perceptual / discriminator)")
elif LOSS_MODE == "adv_no":
    perc_name = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"
    if PERCEPTUAL_WARMUP_EPOCHS == 0 and WARMUP_EPOCHS == 0:
        print(f"  loss mode: adv-NO — L1 + {perc_name} + adversarial from epoch 1")
    else:
        print(
            f"  loss mode: adv-NO — L1 through ep {PERCEPTUAL_WARMUP_EPOCHS}, "
            f"+{perc_name} from ep {PERCEPTUAL_WARMUP_EPOCHS + 1}, "
            f"+adv from ep {WARMUP_EPOCHS + 1}"
        )
else:
    print(f"  loss mode: {LOSS_MODE!r}")
if LOSS_SPATIAL_MODE == "full":
    spatial_note = "uniform over whole valid field"
elif LOSS_SPATIAL_MODE == "gaussian_center":
    spatial_note = f"gaussian_center (σ={LOSS_GAUSSIAN_SIGMA}) — state⊙G"
else:
    spatial_note = LOSS_SPATIAL_MODE
print(f"  loss spatial: {spatial_note}")
if VIZ_GAUSSIAN_CIRCLES:
    print(
        f"  gauss-circle viz: r={VIZ_GAUSSIAN_CIRCLE_RADII} | "
        f"zoom inner circles r={VIZ_GAUSSIAN_INNER_CIRCLE_RADII} -> samples_gauss_circles/"
    )

Run directory: Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles
Wrote config:  Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles/train_config_adv_no_uv_16_19_dt96_gauss_l1_circles.json
  data: data/sample_16_19_smallfixwin_uv_dt96 | iD=2 | loss_metric=components
  loss mode: L1-only (no perceptual / discriminator)
  loss spatial: gaussian_center (σ=32.0) — state⊙G
  gauss-circle viz: r=(16.0, 32.0, 64.0) | zoom inner circles r=(32.0, 16.0) -> samples_gauss_circles/


In [4]:
from src.train.fix_window_viz import save_loss_mask_preview

if PREVIEW_LOSS_MASK:
    preview_path = save_loss_mask_preview(
        DATA_PREFIX,
        RUN_DIR / "viz" / "loss_mask_preview.png",
        sample_index=0,
        loss_spatial_mode=LOSS_SPATIAL_MODE,
        loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
        channel_names=CHANNEL_NAMES,
        distance_circle_radii=VIZ_GAUSSIAN_CIRCLE_RADII,
        show=SHOW_INLINE_PLOTS,
    )
    print(f"Loss mask preview: {preview_path}")
else:
    print("Loss mask preview skipped (PREVIEW_LOSS_MASK=False)")

Loss mask preview: Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles/viz/loss_mask_preview.png


In [5]:
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)
log_path = log_dir / f"train_{RUN_NAME}.log"

class _Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

with open(log_path, "w") as log_file:
    old_stdout, old_stderr = sys.stdout, sys.stderr
    sys.stdout = _Tee(old_stdout, log_file)
    sys.stderr = _Tee(old_stderr, log_file)
    try:
        print(f"Foreground training — log: {log_path}")
        print("Stop anytime: Kernel → Interrupt (or ■ button)\n")
        importlib.reload(adv_no_train_module)
        adv_no_train_module.run_adv_no_training(train_config)
    finally:
        sys.stdout = old_stdout
        sys.stderr = old_stderr

print(f"\nLog saved: {log_path}")

Foreground training — log: logs/train_adv_no_uv_16_19_dt96_gauss_l1_circles.log
Stop anytime: Kernel → Interrupt (or ■ button)

Dataset: data/sample_16_19_smallfixwin_uv_dt96
Samples: 1461 | crop 128 x 128
Using device: cuda:7
Loss mode: l1
Channels: iD=2 (u, v)
Reconstruction metric: components (uv L1)
Loss spatial focus: Gaussian center (σ=32.0 px)
lambda_perc: 0.0 | beta: 0.0
Overwriting previous run artifacts in Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles
Checkpoints: Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles
History:     Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles/history/training_history_adv_no_uv_16_19_dt96_gauss_l1_circles.pt
Viz:         Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles/viz (every 5 epoch(s), fixed val indices [57])
Gaussian loss ACTIVE: state⊙G and |pred−target|⊙G | σ=32.0px | G(center)=2.794 G(edge)=0.390 ≈7.2× at center | saved Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles/loss_gaussian_weight.pt
  G on circles: 16px=2.447,

/opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/src/train/fix_window_viz.py:266: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc="upper right", fontsize=8)


epoch 1 | val uv L1 0.966838 | (improvement of inf)
Epoch 0001 [L1-only] | Train: 1.043013 (uv L1 1.043013, perc 0.000000, adv 0.000000, D 0.000000) | Val uv L1: 0.966838 (perc 0.000000) | G LR: 2.00e-04
  updated curves -> Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles/viz/curves
epoch 2 | val uv L1 0.930759 | (improvement of 0.036078)
Epoch 0002 [L1-only] | Train: 0.948463 (uv L1 0.948463, perc 0.000000, adv 0.000000, D 0.000000) | Val uv L1: 0.930759 (perc 0.000000) | G LR: 2.00e-04
  updated curves -> Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles/viz/curves
epoch 3 | val uv L1 0.889186 | (improvement of 0.041573)
Epoch 0003 [L1-only] | Train: 0.912573 (uv L1 0.912573, perc 0.000000, adv 0.000000, D 0.000000) | Val uv L1: 0.889186 (perc 0.000000) | G LR: 2.00e-04
  updated curves -> Models/adv_no/adv_no_uv_16_19_dt96_gauss_l1_circles/viz/curves
epoch 4 | val uv L1 0.877804 | (improvement of 0.011383)
Epoch 0004 [L1-only] | Train: 0.888531 (uv L1 0.888531, perc 0.000000, a